1) EDA : size, info, shape 등

2) 전체 테이블 데이터 자료를 기준으로, 원하는 값(행&열)을 어떻게 찾아오고, 결측치는 어떻게 대체할 것인가 : isnull
> 전체 열 가운데, 국가 별 필요한 값만 추출, 결측치를 제거

3) 전체 테이블 내 특정 컬럼을 기준으로 그룹화 하는 방법 & 그룹화된 상태에서 특정 조건에 해당되는 값만 조회
> 특정 국가는 한 번만 사용되면 되기 때문에 그룹 방법 알아야 됨!

4) 특정 컬럼 값 타입 변경
> 동적인 형태의 자료로 변경하려고 할 때, 정수의 형태를 띄고 있어야 하는 경우가 발생할 것이기 때문!

5) 서로 다른 데이터 프레임 간 병합하는 방법
> 국가 별 코로나 확진자 수에 대한 추이를 조회하려고 한다! : 그렇다면 2020-01-01 ~ 2023-12-31 : 하나의 데이터 프레임 안에 각 개별 컬럼으로 존재해야 됨!
> 지금은 각 날짜의 데이터는 별도의 독립적인 csv 파일로 되어 있음 / 가지고 와서 하나의 데이터 프레임으로 쭉 연결시켜야 됨!
> 각 국가 별 이름, 확진, 사망, 회복 데이터 + 국가별 축약 키워드 : 이 때도 하나의 데이터 프레임 안에 모든 데이터들이 개별 컬럼으로 존재해야 됨!

In [7]:
import pandas as pd

df1 = pd.DataFrame({
    "id": [1, 2, 3],
    "customer_id": [1, 2, 3],
    "customer_name": ["Reo", "Rima", "Niki"]
})

df1

,id,customer_id,customer_name
0,1,1,Reo
1,2,2,Rima
2,3,3,Niki


In [14]:
df2 = pd.DataFrame({
    "id": [1, 2, 4],
    "order_id": [100, 200, 300],
    "order_date": ["2021-01-21", "2021-02-03", "2020-10-10"]
})

df2

,id,order_id,order_date
0,1,100,2021-01-21
1,2,200,2021-02-03
2,4,300,2020-10-10


* 서로 다른 데이터 프레임 2개를 생성! > 한 개의 프레임으로 병합을 하고자 함!
* 병합의 방식이 2가지!!!
    1) 상, 하
    2) 좌, 우

In [15]:
# concat 상, 하
pd.concat([df1, df2], axis=0) 

# 서로 다른 데이터 프레임을 상, 하 기준으로 합치는 것! --> 동일한 컬럼명을 가지고 있는 경우에만 정상적으로 값을 찾아오고, 그렇지 않은 경우 Nan으로 대체
# axis = 0 디폴트 값! --> 상하 병합 / axis = 1 --> 좌우 병합

,id,customer_id,customer_name,order_id,order_date
0,1,1.0,Reo,NaN,NaN
1,2,2.0,Rima,NaN,NaN
2,3,3.0,Niki,NaN,NaN
0,1,NaN,NaN,100.0,2021-01-21
1,2,NaN,NaN,200.0,2021-02-03
2,4,NaN,NaN,300.0,2020-10-10


In [17]:
pd.concat([df1, df2], axis=1) 

,id,customer_id,customer_name,id,order_id,order_date
0,1,1,Reo,1,100,2021-01-21
1,2,2,Rima,2,200,2021-02-03
2,3,3,Niki,4,300,2020-10-10


In [16]:
pd.merge(df1, df2) # 공통 요소 = 컬럼을 기준으로 좌, 우 병합이 되어 있다 = INNER JOIN # ON을 작성하지 않았어도 공통 컬럼명

,id,customer_id,customer_name,order_id,order_date
0,1,1,Reo,100,2021-01-21
1,2,2,Rima,200,2021-02-03


In [18]:
pd.merge(df1, df2, on="id") # SQL, JOIN 특정 테이블 간 공통 컬럼 내 ON 전치사를 활용

,id,customer_id,customer_name,order_id,order_date
0,1,1,Reo,100,2021-01-21
1,2,2,Rima,200,2021-02-03


In [19]:
pd.merge(df1, df2, on="id", how = "inner")

,id,customer_id,customer_name,order_id,order_date
0,1,1,Reo,100,2021-01-21
1,2,2,Rima,200,2021-02-03


In [20]:
pd.merge(df1, df2, on="id", how = "outer") # left right 모두 살린 경우

,id,customer_id,customer_name,order_id,order_date
0,1,1.0,Reo,100.0,2021-01-21
1,2,2.0,Rima,200.0,2021-02-03
2,3,3.0,Niki,NaN,NaN
3,4,NaN,NaN,300.0,2020-10-10


In [21]:
pd.merge(df1, df2, on="id", how = "left") # left outer join

,id,customer_id,customer_name,order_id,order_date
0,1,1,Reo,100.0,2021-01-21
1,2,2,Rima,200.0,2021-02-03
2,3,3,Niki,NaN,NaN


In [ ]:
pd.merge(df1, df2, on="id", how = "right") # right outer join

지금까지는 서로 다른 데이터 프레임을 하나로 병합하는 조건 : 특정 컬럼을 기준으로!!!
인덱스를 기준으로 병합

In [22]:
df1 = df1.set_index("id")
df1

,customer_id,customer_name
id,,
1,1,Reo
2,2,Rima
3,3,Niki


In [23]:
df2 = df2.set_index("id")
df2

,order_id,order_date
id,,
1,100,2021-01-21
2,200,2021-02-03
4,300,2020-10-10


In [24]:
pd.merge(df1, df2, left_index=True, right_index=True)
# 인덱스를 기준으로 inner join / 양쪽에 존재하는 참인 요소들만 찾아온 것!

,customer_id,customer_name,order_id,order_date
id,,,,
1,1,Reo,100,2021-01-21
2,2,Rima,200,2021-02-03


In [25]:
pd.merge(df1, df2, how="outer", left_index=True, right_index=True)
# 인덱스를 기준으로 병합을 하되, left, right 모두 outer로 생존!

,customer_id,customer_name,order_id,order_date
id,,,,
1,1.0,Reo,100.0,2021-01-21
2,2.0,Rima,200.0,2021-02-03
3,3.0,Niki,NaN,NaN
4,NaN,NaN,300.0,2020-10-10


In [26]:
pd.merge(df1, df2, how="left", left_index=True, right_index=True)
# 인덱스를 기준으로 병합을 하되, left outer 생존

,customer_id,customer_name,order_id,order_date
id,,,,
1,1,Reo,100.0,2021-01-21
2,2,Rima,200.0,2021-02-03
3,3,Niki,NaN,NaN


In [27]:
pd.merge(df1, df2, how="right", left_index=True, right_index=True)
# 인덱스를 기준으로 병합을 하되, right outer 생존

,customer_id,customer_name,order_id,order_date
id,,,,
1,1.0,Reo,100,2021-01-21
2,2.0,Rima,200,2021-02-03
4,NaN,NaN,300,2020-10-10
